![Week 6 Day 5](../assets/banners/w6d5.png)

# Week 6 Day 5 - Responsible AI Grading, the Capstone Showcase and the Final Roadmap

This is it: the last lab of the course. 🎓

Six weeks ago you made your first call to an LLM and asked it to mark Sam's deadlock answer. It gave a mark, and you listed everything that was missing: structure, evidence, course knowledge, handwriting, checks, a human in the loop, an app. This week you ticked off the last item. GradeWise has an API, a dashboard, live progress, a review queue, report cards, chat, and a Docker image that runs it all with one command.

Today is about stepping back. First, the most important lecture of the course: **what it takes to grade responsibly with AI**, and where AI grading should stop. Then the **capstone showcase**: the whole product, end to end, in one cell. Then where to take GradeWise next, how to make it your portfolio piece, the final roadmap, and a thank-you.

![The whole journey](../assets/roadmap/overview.png)

### 🎯 By the end of this lab you will
1. ⚖️ Explain how GradeWise handles fairness, transparency, oversight, privacy and data retention, principle by principle
2. 🛑 State the limits of AI grading plainly, and design around them
3. 🏆 Run the complete capstone (ingest → exam → handwritten scripts → grade → review → report card) in one cell
4. 🧭 Pick your next step: maths and diagrams, more languages, Deep Agents, or an MCP server
5. 💼 Turn GradeWise into a portfolio piece you can demo in five minutes

### 🎬 Videos for this lab

| # | Lecture | Type |
|:-:|---|:-:|
| 1 | Day 5 - Responsible AI Grading: Fairness, Transparency, Privacy | 🎬 |
| 2 | Day 5 - Capstone Showcase: From Upload to Report Card | 🎬 demo |
| 3 | Day 5 - Where Next: Math and Diagrams, Deep Agents, an MCP Server for GradeWise | 🎬 |
| 4 | Day 5 - Make It Your Portfolio Piece | 🎬 |
| 5 | Day 5 - The Final Roadmap and Thank You | 🎬 |

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">Everything today runs <b>offline and free</b> in 🧪 demo mode, including the capstone. Remember what that means: the plumbing is real (uploads, the exam graph and its approval interrupt, background grading, guardrails, the review interrupts, report cards, storage, chat tools), but the "examiner" is a keyword matcher, so <b>the marks mean nothing</b>. To see real marks, set your API key and turn demo mode off in the dashboard. Today's sandbox is <code>output/week6/capstone/</code>.</span>
        </td>
    </tr>
</table>

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv
from IPython.display import HTML, Markdown, display

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f"✅ OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("ℹ️ No OpenAI API Key set. That's fine this week: every cell runs offline in 🧪 demo mode.")
    print("   (To grade for real later, see Part 5 of the setup guide, or setup/SWAP_PROVIDER.md)")

## Part 1: Responsible AI grading

A mark isn't a chatbot reply. It goes on a transcript, decides who passes, shapes how a student sees themselves. If GradeWise gets a mark wrong, a real Sam pays for it. So "responsible AI" here isn't a slide at the end: it's been a design constraint since week 1. Let's make it explicit.

### Start from the rules humans already follow
Asha's CS201 handbook has ten **general marking principles**: the rules every human marker at Northbridge follows. The best test of an AI grader is simple: *does it follow the same rules, and can you show where?*

In [ ]:
import re

from gradewise.config import DATA_DIR, OUTPUT_DIR, REPO_ROOT

handbook = (REPO_ROOT / "scripts" / "knowledge_base_source" / "cs201_course_handbook.md").read_text(encoding="utf-8")
section = handbook.split("## General marking principles", 1)[1].split("\n## ", 1)[0]
principles = {
    int(n): re.sub(r"\*\*", "", text).split(". ")[0] for n, text in re.findall(r"^(\d+)\. (.+)$", section, re.M)
}

HOW_GRADEWISE_HONOURS_IT = {
    1: "Week 4: transcription is verbatim, never 'fixed'; the examiner prompt accepts misspelt terms",
    2: "Week 5: criterion-by-criterion marking; marks are awarded per criterion, not deducted",
    3: "Week 5: the examiner prompt + criteria describe ideas, keywords are 'helpful, not required'",
    4: "Week 5: the examiner prompt caps a bare keyword list at half marks",
    5: "Week 5: in the examiner prompt, but working isn't yet extracted step by step (see Part 3)",
    6: "Week 5: in the examiner prompt, but only as good as the prompt today (see Part 3)",
    7: "Week 5: enforce_rubric() clamps every criterion; Week 6: the review queue caps the input",
    8: "Week 4: crossed-out text is transcribed as such; the grader is told to ignore it",
    9: "Weeks 4-6: low confidence or [illegible] → flagged → the review queue, never a guessed zero",
    10: "Week 5: the injection tripwire strips instructions and flags the script for Asha",
}
rows = "\n".join(f"| {n} | {principles[n]} | {HOW_GRADEWISE_HONOURS_IT[n]} |" for n in sorted(principles))
display(Markdown("| # | Handbook principle | How GradeWise honours it |\n|:-:|---|---|\n" + rows))

All ten are written into the examiner prompt (`gradewise/grading/prompts.py`), and most are backed by code too: verbatim transcription, per-criterion schemas, `enforce_rubric()`, crossed-out text removed before grading, confidence flags and the injection tripwire. Two (method marks and follow-through for calculations) are only as good as the prompt today, because GradeWise doesn't yet read working step by step. Knowing which rules are *enforced* and which are merely *requested* is half of responsible engineering. Now the five pillars, and what GradeWise does for each.

### ⚖️ Fairness: the same answer gets the same mark, whoever wrote it
- **Halo effect.** Each question is graded in its own agent run that only sees that question and that answer (week 5's `Send` fan-out). A brilliant Q3 can't inflate a weak Q4b.
- **Handwriting and language.** Messy handwriting and spelling must not cost marks (principle 1). Faithful transcription keeps *what* was written; the rubric judges the *meaning*. 🚀 A good audit: re-grade the same answer typed neatly vs transcribed from messy handwriting, and from native vs non-native English phrasing. The marks should match.
- **Consistency.** Structured output, optional self-consistency runs (temperature alone isn't a guarantee: some models ignore it), and the **golden set** (week 5) to measure agreement with a human (QWK) rather than hoping for it. Measure it **per subgroup** too: an average can hide a group that's marked harshly.

### 🔍 Transparency: every mark can explain itself
- Every mark is broken down **criterion by criterion**, each with a justification and the **evidence** it relied on (a citation into the course notes).
- The report card shows the student *why*, not just *how much*. Teacher changes are recorded with their history (`teacher override 1.5 → 3`).
- LangSmith traces (week 5) let you replay exactly what the model saw and said for any mark.

### 🧑‍🏫 Oversight: the human has the last word
- Uncertain cases **pause the graph** (`interrupt()`): low transcription confidence, low grader confidence, injection attempts, blanks. Nothing uncertain reaches a student without a teacher.
- The review queue shows the answer, the reason, the reasoning and the evidence, and lets Asha override any mark. "Auto-approve" exists for demos and is **off by default**.
- 🚀 Track the override rate. If teachers override 30% of flags, the thresholds or the grader need work; if they override 0%, ask whether they're really reading them.

### 🔒 Privacy: student work is personal data
- **Data minimisation.** GradeWise works with student IDs, not names. Names, if you need them, stay in the school's own system.
- **Know where data goes.** Every model call sends a student's answer to a provider. Check the provider's data-use and retention terms, use business/API terms that exclude training, pick a region that fits the law (GDPR in Europe, FERPA in the US, and local equivalents), and get the school's sign-off. A local model (Ollama, week 1) keeps everything on-premises.
- **Never** put real students' work in a public demo. The synthetic class exists for exactly that.

### 🗑️ Data retention: keep it only as long as you need it
Uploads, transcriptions, checkpoints and report cards pile up in `output/`. Agree a policy with the school (for example: delete scripts and checkpoints at the end of term, keep final marks as the school requires) and **automate** it. Here's what such a job looks like, as a dry run on this week's sandboxes:

In [ ]:
import time
from datetime import datetime, timedelta


def retention_report(folder, older_than_days: float, *, delete: bool = False) -> None:
    """List (and optionally delete) files older than the retention period. ⚠️ Dry run unless delete=True."""
    cutoff = time.time() - older_than_days * 86400
    old = [p for p in folder.rglob("*") if p.is_file() and p.stat().st_mtime < cutoff]
    kinds: dict[str, list] = {}
    for p in old:
        kinds.setdefault(p.suffix or "(no extension)", []).append(p)
    when = datetime.now() - timedelta(days=older_than_days)
    print(f"🗂️ {folder} · {len(old)} file(s) last changed before {when:%Y-%m-%d %H:%M}")
    for suffix, files in sorted(kinds.items(), key=lambda kv: -len(kv[1])):
        size = sum(f.stat().st_size for f in files) / 1024
        print(f"   {suffix:<15} {len(files):>4} file(s) · {size:,.0f} KB")
    if delete:
        for p in old:
            p.unlink()
        print("   🗑️ deleted")
    else:
        print("   (dry run: nothing deleted)")


week6_output = OUTPUT_DIR / "week6"
week6_output.mkdir(parents=True, exist_ok=True)
retention_report(week6_output, older_than_days=0)  # 💡 0 days = "everything", just to see what piles up

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">Should GradeWise ever release a mark to a student without a teacher approving it? Think about a weekly low-stakes quiz versus a final exam, and about what "approving" should mean when there are 300 scripts. Is there a version where most marks are released automatically and that is still responsible? What would have to be true about the evidence (golden-set QWK, override rates, subgroup checks) before you'd sign off on it?</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;"><b>The explicit limits of AI grading.</b> Say these out loud to anyone who will use GradeWise:<br/>
• <b>It is an assistant, not an examiner.</b> For any mark that matters (finals, certification, progression) a qualified human must review and own the decision.<br/>
• <b>Transcription errors become grading errors.</b> A misread word can cost a mark; confidence scores reduce this, they don't remove it.<br/>
• <b>Rubric-bound.</b> A brilliant answer the rubric didn't anticipate may be under-marked. That's what the review queue is for.<br/>
• <b>Not yet good at</b> diagrams, maths working, or code on paper, and only tested in English.<br/>
• <b>Models change.</b> A provider update can shift marks overnight: pin model versions and re-run the golden set before each term.<br/>
• <b>Adversaries exist.</b> The injection tripwire catches the obvious attempts, not every possible one.<br/>
• <b>Bias can hide in averages.</b> Check agreement per subgroup, not just overall.</span>
        </td>
    </tr>
</table>

## Part 2: The capstone showcase: from upload to report card

Here is everything you built, in one cell, through the same API the dashboard uses. A fresh GradeWise starts in an empty folder; Asha uploads the course notes and the exam pack and approves the extraction; five **handwritten** answer-sheet PDFs (the synthetic class, including Sam, `student_05`) are graded in the background; Asha reviews every flag (and overrides one mark); and Sam's report card appears below.

Run it, and enjoy it. You built every piece of this. 🎉

In [ ]:
import html
import shutil

from fastapi.testclient import TestClient

from gradewise.api.main import create_app
from gradewise.console import banner, step, success

banner("🏆 GradeWise: the capstone", "upload → extract → approve → transcribe → grade → review → report card")
SHOWCASE = OUTPUT_DIR / "week6" / "capstone"
shutil.rmtree(SHOWCASE, ignore_errors=True)
api = TestClient(create_app(SHOWCASE, demo=True))  # 🧪 a brand-new GradeWise, offline fakes, empty folder
started = time.perf_counter()


def until_stopped(batch_id: str) -> dict:
    while (batch := api.get(f"/batches/{batch_id}").json())["status"] not in ("paused", "done", "failed"):
        time.sleep(0.2)
    return batch


step("1/6 📚 Asha uploads the CS201 course notes (week 2: ingestion)")
kb = api.post("/knowledge-bases/cs201/documents", data={"include_course_notes": "true"}).json()

step("2/6 📝 …and the exam pack, which is extracted and waits for her approval (weeks 1 & 3)")
pack = DATA_DIR / "exams" / "cs_midterm"
exam_files = {
    k: (f"{k}.pdf", (pack / f"{k}.pdf").read_bytes(), "application/pdf")
    for k in ("question_paper", "marking_scheme", "methodology")
}
extracted = api.post("/exams", data={"exam_id": "cs_midterm"}, files=exam_files).json()
approved = api.post("/exams/cs_midterm/approve", json={"approved": True}).json()

step("3/6 ✍️ Five handwritten scripts are transcribed and graded in the background (weeks 4 & 5)")
batch_id = api.post("/batches", data={"exam_id": "cs_midterm", "use_synthetic": "true", "students": "5"}).json()[
    "batch_id"
]
batch = until_stopped(batch_id)

step("4/6 🧑‍🏫 Asha works through the review queue (week 6)")
flags, override = 0, None
while batch["status"] == "paused":
    pending = batch["pending"]
    flags += len(pending["items"])
    changes = {}
    if override is None and pending["type"] == "human_review":
        item = pending["items"][0]
        changes = {item["question_id"]: min(item["max_marks"], item["awarded"] + 1)}  # she sees one more mark
        override = (pending["student_id"], item["question_id"])
    api.post(
        f"/batches/{batch_id}/review",
        json={"student_id": pending["student_id"], "overrides": changes, "approved": True},
    ).raise_for_status()
    batch = until_stopped(batch_id)

step("5/6 📊 The class results (numbers from code, not from a model)")
overview = api.get("/reports/cs_midterm").json()

step("6/6 📄 Sam's report card")
sam = "student_05"
card_html = api.get(f"/reports/cs_midterm/{sam}", params={"format": "html"}).text

success(
    "Capstone complete",
    {
        "Knowledge base": f"{kb['chunks']} chunks from {len(kb['files'])} PDFs",
        "Exam": f"{extracted['summary']['question_count']} questions · {approved['status']}",
        "Batch": f"{batch_id} · {batch['status']} · {batch['events']} progress events",
        "Human review": f"{flags} flagged answer(s) reviewed · 1 override on {override[0]} {override[1]}",
        "Class": f"{overview['count']} students · mean {overview['mean_percent']}% · bands {overview['bands']}",
        "Time": f"{time.perf_counter() - started:.1f} s, $0.00",
    },
    next_step="Scroll down: Sam's report card",
)
frame = f'<iframe srcdoc="{html.escape(card_html)}" style="width:100%;height:820px;border:0"></iframe>'
display(HTML(f"<div>{frame}</div>"))

🎉 **That's GradeWise.** Six weeks of work, end to end, in one cell.

Now let's audit it the way an exam board would. Is every mark on Sam's card explained? Is every awarded mark backed by evidence? Can we see what the humans changed?

In [ ]:
card = api.get(f"/reports/cs_midterm/{sam}").json()
criteria = [c for r in card["results"] for c in r["criteria"]]
awarded = [c for c in criteria if c["awarded"] > 0]
print(
    f"🔍 Transparency audit for {sam}: {card['total']:g}/{card['max_total']:g} "
    f"({card['percentage']:g}%), band {card['grade_band']}"
)
justified = sum(bool(c["justification"]) for c in criteria)
print(f"   criteria with a justification     : {justified}/{len(criteria)}")
print(f"   awarded criteria citing evidence  : {sum(bool(c['evidence']) for c in awarded)}/{len(awarded)}")
print(f"   marks above the maximum           : {sum(r['awarded_marks'] > r['max_marks'] for r in card['results'])}")
print(f"   answers a teacher reviewed        : {sum(bool(r['review_reason']) for r in card['results'])}")
for r in card["results"]:
    if r["review_reason"]:
        print(f"      {r['question_id']}: {r['review_reason']}")

answer = api.post(
    "/ask", json={"question": "Which question did the class struggle with most?", "exam_id": "cs_midterm"}
)
display(
    Markdown("**💬 Asha asks: which question did the class struggle with most?**\n\n" + answer.json()["answer"][:700])
)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/gradewise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">GradeWise</h2>
            <span style="color:#7c5cff;">🏁 <b>Milestone M6 reached: GradeWise is shipped.</b> The capstone cell used only the public API (<code>gradewise/api/</code>), which drives the exam graph (<code>gradewise/exam/</code>), the ingestion pipeline (<code>gradewise/ingestion/</code>), the orchestrator (<code>gradewise/orchestrator.py</code>) with transcription (<code>gradewise/transcription/</code>), grading (<code>gradewise/grading/</code>) and reports (<code>gradewise/reports/</code>). The same flow runs in the browser with <code>docker compose up --build</code> or the two-terminal commands in <code>week6/project/README.md</code>, and <code>uv run python -m pytest tests/test_api.py -q</code> checks it in CI.</span>
        </td>
    </tr>
</table>

## Part 3: Where next?

GradeWise is a foundation, not a finish line. Four directions, from "a weekend" to "a startup":

### 🧮 Maths and diagrams
Principles 5 and 6 (method marks, follow-through) are where GradeWise is weakest today. Ideas:
- Ask the vision model to transcribe working **line by line** as structured steps (`list[Step]`), and grade each step against a marking scheme with method and answer marks.
- Check arithmetic with **code, not the model**: parse each step and verify it with `sympy` (numbers from code, words from the model: the same rule as the results agent).
- For diagrams (a process-state diagram, a subnet sketch), have the vision model describe them as a graph of labelled nodes and edges, then grade that structure against the criteria. Flag low-confidence diagram readings for review.

### 🌍 More languages
- Use **multilingual embeddings** so a French answer can retrieve English course notes (or ingest the notes in both languages).
- Measure **CER per script** (Latin, Devanagari, Arabic…) on a golden set before you trust transcription in that language.
- Write feedback in the student's language, but keep the marking rubric and evidence in the language the examiner reads.

### 🤖 Deep Agents
LangChain's [Deep Agents](https://docs.langchain.com/oss/python/deepagents/overview) (`deepagents`) add planning, a virtual file system and sub-agents on top of `create_agent`. A natural GradeWise use is a **moderation agent** that plans its own work for a whole cohort: "sample 10% of scripts, re-mark the borderline ones with a sub-agent, compare with the first marker, write a moderation report to a file." It isn't installed in the course environment (`uv add deepagents` on your own branch), and the sketch looks like this:

```python
from deepagents import create_deep_agent

moderator = create_deep_agent(
    model=grading_model(),
    tools=[class_statistics, get_student_report, regrade_answer],  # your week 5 tools + a re-marker
    system_prompt="You are the CS201 moderator. Plan, sample, re-mark borderline answers, write a report.",
)
```

### 🔌 An MCP server for GradeWise
The **Model Context Protocol** lets any MCP-aware assistant (Claude Desktop, Cursor, VS Code…) call your tools. GradeWise already has the perfect tool layer: its API. An MCP server is a thin wrapper around a few functions like these, which already work today:

In [ ]:
# 💡 Candidate MCP tools: small, typed, documented functions over the GradeWise API.
def list_exams() -> list[dict]:
    """List the exams GradeWise can grade against."""
    return [{"exam_id": e["exam_id"], "title": e["title"]} for e in api.get("/exams").json()]


def class_report(exam_id: str) -> dict:
    """Class statistics for one exam: mean, median, grade bands and the hardest questions."""
    r = api.get(f"/reports/{exam_id}").json()
    return {k: r[k] for k in ("count", "mean_percent", "median_percent", "bands")} | {"hardest": r["questions"][:3]}


def ask_about_results(exam_id: str, question: str) -> str:
    """Ask the results agent a question about an exam's report cards."""
    return api.post("/ask", json={"exam_id": exam_id, "question": question}).json()["answer"]


print(list_exams())
print(class_report("cs_midterm"))

Turning them into an MCP server is a few decorators with the official Python SDK (`uv add mcp` on your own branch; it's not in the course environment):

```python
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("GradeWise")
mcp.tool()(list_exams)          # the docstring becomes the tool description the assistant reads
mcp.tool()(class_report)
mcp.tool()(ask_about_results)

if __name__ == "__main__":
    mcp.run()                   # stdio: add it to Claude Desktop / Cursor as an MCP server
```

⚠️ Expose **read-only** tools first. A tool that changes marks should go through the same review interrupt as everything else, never straight to the database.

## Part 4: Make it your portfolio piece

GradeWise is a serious project: agentic RAG, LangGraph orchestration, vision, evaluation, an API, a UI and Docker. Here's how to make sure a hiring manager *sees* that in five minutes.

**✅ The portfolio checklist**
- [ ] **Your own fork**, with a README that opens with one sentence ("An AI teaching assistant that grades handwritten exams with evidence, and asks a teacher before anything uncertain"), a GIF or screenshots of the dashboard, and the architecture diagram
- [ ] **A live demo link** (Hugging Face Spaces, demo mode, Day 4) and a 2-minute video walkthrough
- [ ] **Your own twist**: a new subject (this week's exercise), a better grader, maths support, an MCP server… Something that's clearly *yours*
- [ ] **Numbers**: your grader's QWK against the golden set vs the baseline, cost per script, time per batch. Numbers are what make a project credible
- [ ] **A "Responsible AI" section**: what it can't do, where the human is, how data is handled (Part 1 of today)
- [ ] **Green tests and CI**: `uv run python -m pytest` passing, ideally in a GitHub Action
- [ ] **A short write-up**: a blog post or LinkedIn post on one hard problem you solved (faithful transcription? prompt injection? interrupts?)
- [ ] **Share it** in `community_contributions/`, and tag the course: we'd love to see it

Let's tick one box right now: the API's test suite, the same one a CI job would run:

In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_api.py", "-q", "-p", "no:cacheprovider"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)
print(result.stdout.strip().splitlines()[-1] if result.stdout.strip() else result.stderr[-800:])
print("✅ green: put that badge in your README!" if result.returncode == 0 else "❌ something failed: see above")

💡 Interviewers love "what would you do differently?" Have answers ready: a task queue instead of threads (`gradewise/api/jobs.py` explains), Postgres instead of SQLite, per-subgroup fairness metrics, a moderation agent… Knowing the limits of your own system is the strongest signal of seniority.

## Part 5: The final roadmap

Let's look back at the whole road, and count what you built:

In [ ]:
import yaml

package = [p for p in (REPO_ROOT / "gradewise").rglob("*.py")]
lines = sum(len(p.read_text(encoding="utf-8").splitlines()) for p in package)
tests = sum(p.read_text(encoding="utf-8").count("\ndef test_") for p in (REPO_ROOT / "tests").glob("test_*.py"))
labs = sorted(p for p in REPO_ROOT.glob("week*/day*.ipynb"))
quiz_questions = sum(
    len(yaml.safe_load(p.read_text(encoding="utf-8"))["questions"]) for p in (REPO_ROOT / "quizzes").glob("*.yaml")
)

success(
    "🎓 GradeWise, by the numbers",
    {
        "Weeks": "6",
        "Lab notebooks": str(len(labs)),
        "Package": f"{len(package)} Python files, {lines:,} lines in gradewise/",
        "Tests": f"{tests} test functions, all offline",
        "Quiz questions": str(quiz_questions),
        "Handbook principles": "10 of 10 in the examiner prompt, most also enforced in code",
        "Cost of today's capstone": "$0.00",
    },
)

![Diagram](../assets/diagrams/b57f2c35068d.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
flowchart LR
  W1["✅ W1<br/>LLMs & structured output<br/>exam → JSON"] --> W2["✅ W2<br/>RAG foundations<br/>the knowledge base"]
  W2 --> W3["✅ W3<br/>Agents & LangGraph<br/>exam graph + approval"]
  W3 --> W4["✅ W4<br/>Agentic RAG & vision<br/>evidence + handwriting"]
  W4 --> W5["✅ W5<br/>Grading & evaluation<br/>report cards, QWK"]
  W5 --> W6["✅ W6<br/>Ship it<br/>API, UI, Docker, deploy"]
  W6 --> YOU(["🚀 Your version"])
  classDef done fill:#7c5cff,color:#fff,stroke:#7c5cff
  classDef you fill:#22c55e,color:#fff,stroke:#22c55e
  class W1,W2,W3,W4,W5,W6 done
  class YOU you
```

</details>


| Week | You learned | You built |
|:-:|---|---|
| 1 | LLMs, tokens and cost; prompting; structured output with Pydantic; multimodal input; tools | The exam pack → a validated `ExamPaper` |
| 2 | Embeddings, chunking, Chroma, hybrid search, reranking, MRR and nDCG | An idempotent, evaluated knowledge base |
| 3 | Agents, `create_agent`, middleware, `StateGraph`, `Send`, checkpoints, `interrupt()` | The exam graph with admin approval |
| 4 | Agentic RAG, grounded citations, OCR vs vision LLMs, faithful transcription | The evidence retriever and the transcription subgraph |
| 5 | LLM-as-grader, guardrails, prompt-injection defence, report cards, MAE / QWK / CER | Parallel evidence-based grading, report cards, the orchestrator, evaluation |
| 6 | FastAPI, background jobs, SSE, Streamlit, Docker, deployment, responsible AI | **A shipped product** |

Remember Week 1 Day 1's table, "Why this isn't GradeWise (yet)"? Every row is done: structured output, course knowledge, checks and retries, human approval, handwriting, evidence, consistency measured against humans, and an app in Docker.

## Recap, and where we are heading

Today you:
- ⚖️ mapped the handbook's ten marking principles onto GradeWise, and walked through fairness, transparency, oversight, privacy and retention
- 🛑 stated the limits of AI grading plainly
- 🏆 ran the whole capstone in one cell, and audited Sam's report card for transparency
- 🧭 explored four next steps: maths and diagrams, more languages, Deep Agents and an MCP server
- 💼 got a portfolio checklist, and a green test suite to show for it

**Where are you heading?** Wherever you like. You can build agentic RAG systems that read documents, follow rules, show their evidence, keep a human in control and ship as a real product. That's a rare and valuable skill set.

📝 **Quiz 6** (6 questions) and the **Final quiz** (20 questions, all six weeks) are waiting for you on Udemy.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Exercise</h2>
            <span style="color:#f59e0b;">The final exercise is <code>week6/week6_EXERCISE.ipynb</code>: <b>ship your own variant of GradeWise.</b> Pick a new subject (biology, history, economics, a programming course…), write a few pages of course notes and a short exam with a marking scheme, run it through the API end to end (ingest → approve → grade → review → report cards → chat), and package it for <code>community_contributions/</code> so other learners can try it. It's all offline in demo mode, and a complete worked example is in <code>week6/solutions/week6_EXERCISE_SOLUTION.ipynb</code>.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/thankyou.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ec4899;">Thank you</h2>
            <span style="color:#ec4899;"><b>Thank you.</b> Six weeks ago GradeWise was one prompt and a mark you couldn't trust. Today it reads handwriting faithfully, grades with evidence, resists manipulation, measures itself against a human examiner, asks a teacher before anything uncertain becomes a mark, and runs anywhere with one command. <i>You</i> built that, one small cell at a time, through API errors, Windows quirks and at least one "why is it paused?!" moment. That persistence is the real skill, and it's yours to keep.<br/><br/>Thank you for trusting this course with your time. If it helped you, the kindest things you can do are to leave an honest review on Udemy, share what you built in <code>community_contributions/</code>, and tell someone who'd enjoy it. I read every review and every contribution, and I'd genuinely love to see your version of GradeWise and hear where it takes you. Asha, Sam and the whole Northbridge Institute class say thank you too. 🎓<br/><br/>Now go and build something wonderful. 🚀</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/resources.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#3b82f6;">Resources</h2>
            <span style="color:#3b82f6;">• Responsible AI in education: <a href="https://www.unesco.org/en/digital-education/artificial-intelligence">UNESCO: AI in education</a> · <a href="https://www.ed.gov/sites/ed/files/documents/ai-report/ai-report.pdf">US Dept. of Education: AI and the Future of Teaching and Learning</a><br/>• Model Context Protocol: <a href="https://modelcontextprotocol.io/">modelcontextprotocol.io</a> · Python SDK: <a href="https://github.com/modelcontextprotocol/python-sdk">github.com/modelcontextprotocol/python-sdk</a><br/>• Deep Agents: <a href="https://docs.langchain.com/oss/python/deepagents/overview">docs.langchain.com/oss/python/deepagents</a><br/>• LangChain Academy (free courses): <a href="https://academy.langchain.com/">academy.langchain.com</a><br/>• This repo's <code>RESOURCES.md</code> for everything else we used along the way</span>
        </td>
    </tr>
</table>